# Exploratory Data Auditing: Cleaning & Schema Normalization
[Brian C. Keegan, Ph.D.](http://www.brianckeegan.com)
May 2026

Released under an [MIT License](https://opensource.org/licenses/MIT).

This notebook is the **reproducibility trust contract** for the cleaning half
of the *Exploratory Data Auditing* analysis. It reads every quarterly U.S.
House *Statement of Disbursements* file under `data/`, normalizes four distinct
historical schemas into one common schema, validates that schema with
integrity checks, and writes a single cleaned artifact (`all_disbursements.csv`)
that `analysis.ipynb` consumes. No schema or datetime work happens downstream.

**Data provenance.** Quarterly disbursement files are published by the
[U.S. House of Representatives Statement of Disbursements](https://disbursements.house.gov/)
(the canonical primary source); the
[ProPublica mirror](https://projects.propublica.org/represent/expenditures)
is a secondary source. The raw files (~1 GB) are **not committed**; run
`python scripts/fetch_data.py` to retrieve them into `data/`. Only the cleaned
output is tracked (via Git LFS).

**Scope & limitations.** The 62 source files (2010Q1–2024Q4 CSV plus
2025Q1–2025Q2 XLSX) are all read, but the **2010 quarters are excluded** from
the cleaned schema: the 2010Q1–Q2 raw files cram the transaction date, posting
code, record id, and payee into a single fixed-width `PAYEE` field (a
pre-Schema-1 layout the upstream source never normalized), and the original
analysis likewise excluded all of 2010. The cleaned universe therefore begins
**2011Q1** and runs through **2025Q2**, a consistent year boundary. From
2023Q1 onward the source dropped `BIOGUIDE_ID`; it is reconstructed by
name-matching `ORGANIZATION` against `propublica_members.csv`, so member-level
analyses span the full 2011–2025 universe (non-member offices and vacant
district placeholders stay null). The transaction `DATE` is the true posting date where the source
provides one, null otherwise; `PERIOD_DATE` (= `DATE` else `START DATE`) is
provided for time-series binning.

**Pinned headline numbers** (asserted near the end; a change here is a loud,
intentional signal that the source data shifted): 58 cleaned quarters,
6,233,106 raw rows, 6,040,756 cleaned rows, 192,350 subtotal/total rows
removed, grand-total `AMOUNT` ≈ \$20,174,379,954.18.

## Imports

Single imports cell — no other cell imports. The notebook does no plotting and
no network I/O; biographical retrieval lives in `scripts/fetch_biographical.py`.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 100
warnings.filterwarnings("ignore")

## Configuration & provenance

One folder of quarterly files, four historical schema eras plus two
single-quarter transitional schemas. The era → adapter map below is the
authoritative dispatch; each adapter is documented in its own section.

| Era | Quarters | Shape |
|---|---|---|
| *(excluded)* | 2010Q1–2010Q4 | pre-Schema-1; fields crammed into `PAYEE` — not validly normalizable |
| **A** Schema 1 | 2011Q1–2016Q3 | `BIOGUIDE_ID`/`OFFICE`; year-less `DATE` (`MM-DD` or `DD-Mon`) |
| **B** Schema 2A | 2016Q4–2022Q3 (excl. 2018Q2) | `PROGRAM`/`SORT SEQUENCE` appear; 2016Q4–2017Q1 have no `SORT SEQUENCE`; **2017Q2 is positionally shifted** |
| **C** Schema 2B-pre | 2018Q2 only | distinct mini-schema (`SORT SUBTOTAL DESCRIPTION`, `TRANSACTION DATE`, `DATA SOURCE`, `DOCUMENT`) |
| **D** Schema 2B | 2022Q4 only | major overhaul; `PERFORM START/END DT`; `DATE` ISO-populated |
| **E** Schema 3 | 2023Q1–2024Q4 (CSV) + 2025Q1–Q2 (XLSX) | **no `BIOGUIDE_ID`**; `ORGANIZATION`/`VENDOR NAME`/`DESCRIPTION` |

Source: [U.S. House Statement of Disbursements](https://disbursements.house.gov/).

In [2]:
DATA_DIR = Path("data")
OUTPUT_PATH = Path("all_disbursements.csv")

# 2010 is read but excluded: 2010Q1-Q2 cram date/code/id/payee into one
# fixed-width PAYEE field (pre-Schema-1); the original analysis dropped all
# of 2010. Cleaned universe is a consistent 2011Q1..2025Q2 boundary.
EXCLUDED_QUARTERS = ["2010Q1", "2010Q2", "2010Q3", "2010Q4"]
EXPECTED_QUARTERS = [
    f"{y}Q{q}" for y in range(2011, 2025) for q in range(1, 5)
] + ["2025Q1", "2025Q2"]
assert len(EXPECTED_QUARTERS) == 58


def schema_era(quarter: str) -> str:
    y, q = int(quarter[:4]), int(quarter[5])
    if quarter == "2018Q2":
        return "C"
    if quarter == "2022Q4":
        return "D"
    if y >= 2023:
        return "E"
    if (y, q) <= (2016, 3):
        return "A"
    return "B"

## File discovery

Glob `data/` for the quarterly disbursement files (CSV 2010–2024, XLSX 2025).
All 62 source files must be present; the four 2010 quarters are recognized but
deliberately excluded from the cleaned universe (documented above). The
pipeline then processes exactly the 58 expected quarters.

In [3]:
def find_files() -> dict:
    files = {}
    for p in sorted(DATA_DIR.iterdir()):
        m = re.match(r"(\d{4}Q\d)-house-disburse-detail", p.name)
        if m and p.suffix in (".csv", ".xlsx"):
            files[m.group(1)] = p
    return files


source_files = find_files()
missing = set(EXPECTED_QUARTERS) - set(source_files)
extra = set(source_files) - set(EXPECTED_QUARTERS) - set(EXCLUDED_QUARTERS)
assert not missing, f"missing quarters: {sorted(missing)}"
assert not extra, f"unexpected files: {sorted(extra)}"
print(f"discovered {len(source_files)} source files; "
      f"excluding {sorted(set(source_files) & set(EXCLUDED_QUARTERS))}; "
      f"cleaning {len(EXPECTED_QUARTERS)} quarters "
      f"{EXPECTED_QUARTERS[0]} .. {EXPECTED_QUARTERS[-1]}")

discovered 62 source files; excluding ['2010Q1', '2010Q2', '2010Q3', '2010Q4']; cleaning 58 quarters 2011Q1 .. 2025Q2


## Common-schema contract

Every adapter returns the 15 `ADAPTER_COLUMNS`; after concatenation four
derived columns are added to yield the 19-column `COMMON_COLUMNS` contract.
`validate_schema` is asserted by every adapter and again on the final frame,
so any column/dtype drift fails the run loudly.

| Column | Source / notes |
|---|---|
| `YEAR-QUARTER` | filename key (primary join key) |
| `YEAR`,`QUARTER` | from key (calendar, not the raw "FISCAL YEAR" text) |
| `TERM_QUARTER` | Congress-derived position 1–8 |
| `BIOGUIDE_ID` | nullable; from source pre-2023, reconstructed from `ORGANIZATION` via propublica name-match 2023Q1–2025Q2; null for non-member offices and vacant-district placeholders |
| `OFFICE` | Schema 3: from `ORGANIZATION` |
| `PROGRAM` | nullable; absent pre-2016Q4 |
| `CATEGORY` | Schema 3/2018Q2: from `SORT SUBTOTAL DESCRIPTION`; spacing normalized |
| `PAYEE` | Schema 3: from `VENDOR NAME`; raw form preserved |
| `PURPOSE` | Schema 3: from `DESCRIPTION` |
| `AMOUNT` | float; thousands stripped; trailing-space header fixed |
| `DATE` | true transaction date; null where source has none |
| `PERIOD_DATE` | `DATE` else `START DATE` (best-available, for binning) |
| `DATE_IS_RECONSTRUCTED` | bool: `DATE` null and `PERIOD_DATE` from `START DATE` |
| `START DATE`,`END DATE` | Schema 2B/3: from `PERFORM START/END DT` |
| `TRANSCODE` | Schema 2B/3/2018Q2: from `DATA SOURCE` |
| `RECORDID` | Schema 2B/3/2018Q2: from `DOCUMENT` |
| `VOUCHER_ID` | `RECIP (orig.)` (S1/2A) or `VENDOR ID` (2B/3) |

In [4]:
ADAPTER_COLUMNS = [
    "YEAR", "QUARTER", "BIOGUIDE_ID", "OFFICE", "PROGRAM", "CATEGORY",
    "PAYEE", "PURPOSE", "AMOUNT", "DATE", "START DATE", "END DATE",
    "TRANSCODE", "RECORDID", "VOUCHER_ID",
]
COMMON_COLUMNS = [
    "YEAR-QUARTER", "YEAR", "QUARTER", "TERM_QUARTER", "BIOGUIDE_ID",
    "OFFICE", "PROGRAM", "CATEGORY", "PAYEE", "PURPOSE", "AMOUNT",
    "DATE", "PERIOD_DATE", "DATE_IS_RECONSTRUCTED", "START DATE",
    "END DATE", "TRANSCODE", "RECORDID", "VOUCHER_ID",
]


def validate_schema(df: pd.DataFrame, quarter: str,
                     columns=ADAPTER_COLUMNS) -> pd.DataFrame:
    missing = [c for c in columns if c not in df.columns]
    extra = [c for c in df.columns if c not in columns]
    assert not missing, f"{quarter}: missing {missing}"
    assert not extra, f"{quarter}: unexpected {extra}"
    return df[columns]

## Raw read helper and shared parsers

Everything is read as `str` first so positional shifts and the year-less date
reconstruction are deterministic. Named all-null columns are **kept** (2017Q2's
`RECORDID` is legitimately all-null and is needed for the shift fix); only
unnamed trailing columns (Schema-1 `,,,,`) are dropped.

`_to_txn_date` carries the domain knowledge from the original notebook's
cells 68–69: Schema-1 / early-2A files store the transaction date *without a
year* (`MM-DD` in 2011-era files, `DD-Mon` in 2013/2016-era files); the year
is reconstructed from the file's calendar quarter. Garbage like `1-10-28` and
out-of-window values are coerced to `NaT`.

In [5]:
def read_raw(path: Path) -> pd.DataFrame:
    if path.suffix == ".xlsx":
        df = pd.read_excel(path, dtype=str, engine="openpyxl")
    else:
        df = pd.read_csv(path, encoding="latin1", low_memory=False,
                          thousands=",", dtype=str)
    df.columns = [str(c).strip() for c in df.columns]
    drop = [c for c in df.columns
            if c == "" or str(c).lower().startswith("unnamed")]
    return df.drop(columns=drop)


def _to_amount(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce")


_DATE_FORMATS = ("%m/%d/%y", "%Y-%m-%d", "%d-%b-%y",
                 "%Y-%m-%d %H:%M:%S", "%m/%d/%Y")


def _to_dt(s: pd.Series) -> pd.Series:
    """Parse via explicit known formats only (fast, no dateutil OOB)."""
    if s is None:
        return pd.Series(pd.NaT, dtype="datetime64[ns]")
    raw = s.astype("string").str.strip()
    raw = raw.mask(raw.isin(["", "nan", "NaN", "NaT", "None"]))
    out = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
    todo = raw.notna()
    for fmt in _DATE_FORMATS:
        if not todo.any():
            break
        parsed = pd.to_datetime(raw[todo], errors="coerce", format=fmt)
        hit = parsed.notna()
        out.loc[parsed.index[hit]] = parsed[hit]
        todo.loc[parsed.index[hit]] = False
    yr = out.dt.year
    return out.mask((yr < 1990) | (yr > 2035))


def _to_txn_date(s: pd.Series, file_year: int) -> pd.Series:
    """Transaction date. Reconstruct the missing year (orig. cells 68-69)."""
    if s is None:
        return pd.Series(pd.NaT, dtype="datetime64[ns]")
    raw = s.astype("string").str.strip()
    raw = raw.mask(raw.isin(["", "nan", "NaN", "NaT", "None", "   "]))
    out = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
    mmdd = raw.str.match(r"^\d{1,2}-\d{1,2}$").fillna(False)        # 02-01
    ddmon = raw.str.match(r"^\d{1,2}-[A-Za-z]{3}$").fillna(False)   # 15-Apr
    if mmdd.any():
        out.loc[mmdd] = pd.to_datetime(
            raw[mmdd] + f"-{file_year}", errors="coerce", format="%m-%d-%Y")
    if ddmon.any():
        out.loc[ddmon] = pd.to_datetime(
            raw[ddmon] + f"-{file_year}", errors="coerce", format="%d-%b-%Y")
    rest = raw.notna() & ~mmdd & ~ddmon
    if rest.any():
        out.loc[rest] = _to_dt(raw[rest])
    yr = out.dt.year
    return out.mask((yr < 1990) | (yr > 2035))


def _drop_total_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Remove subtotal / grand-total rows (orig. cells 89-90)."""
    if "SORT SEQUENCE" in df.columns:
        df = df.loc[~df["SORT SEQUENCE"].isin(
            ["SUBTOTAL", "GRAND TOTAL FOR ORGANIZATION", "GRAND TOTAL"])]
    if "PURPOSE" in df.columns:
        df = df.loc[~df["PURPOSE"].fillna("").str.contains("TOTALS")]
    return df

## Per-schema adapters

One adapter per era. Each takes the raw frame plus the quarter key and returns
a frame conforming to `ADAPTER_COLUMNS`. Every domain-knowledge fix from the
original notebook is preserved here as an explicit, commented, **asserted**
step rather than an out-of-order patch.

### Adapter A — Schema 1 (2010Q1–2016Q3)

`BIOGUIDE_ID`/`OFFICE` present. `RECIP (orig.)` → `VOUCHER_ID`;
`TRANSCODELONG` dropped (it duplicates `TRANSCODE`). Asserts Schema 1 carries
no `SORT SEQUENCE` and no `TOTALS` rows (verified empirically — F5), so the
subtotal filter is provably unnecessary here. The year-less `DATE` is
reconstructed from the file year by `_to_txn_date`.

In [6]:
def adapt_A(raw: pd.DataFrame, quarter: str) -> pd.DataFrame:
    y, q = int(quarter[:4]), int(quarter[5])
    d = raw.rename(columns={"RECIP (orig.)": "VOUCHER_ID"})
    if "TRANSCODELONG" in d.columns:
        d = d.drop(columns="TRANSCODELONG")
    assert "SORT SEQUENCE" not in d.columns, f"{quarter}: unexpected SORT SEQUENCE"
    if "PURPOSE" in d.columns:
        assert not d["PURPOSE"].fillna("").str.contains("TOTALS").any(), \
            f"{quarter}: unexpected TOTALS rows in Schema 1"
    out = pd.DataFrame(index=d.index)
    out["YEAR"] = y
    out["QUARTER"] = f"Q{q}"
    out["BIOGUIDE_ID"] = d.get("BIOGUIDE_ID")
    out["OFFICE"] = d.get("OFFICE")
    out["PROGRAM"] = pd.NA
    out["CATEGORY"] = d.get("CATEGORY")
    out["PAYEE"] = d.get("PAYEE")
    out["PURPOSE"] = d.get("PURPOSE")
    out["AMOUNT"] = _to_amount(d["AMOUNT"])
    out["DATE"] = _to_txn_date(d.get("DATE"), y)
    out["START DATE"] = _to_dt(d.get("START DATE"))
    out["END DATE"] = _to_dt(d.get("END DATE"))
    out["TRANSCODE"] = d.get("TRANSCODE")
    out["RECORDID"] = d.get("RECORDID")
    out["VOUCHER_ID"] = d.get("VOUCHER_ID")
    return validate_schema(out, quarter)

### Adapter B — Schema 2A (2016Q4–2022Q3, excl. 2018Q2)

Heterogeneous era. 2016Q4–2017Q1 have no `SORT SEQUENCE`; 2017Q3+ carry it
with subtotal rows. **2017Q2 is positionally shifted by one column** from the
inserted (mislabeled) `SORT SEQUENCE` onward, plus a spurious trailing
`TOTALS` column — verified against the correctly-formed 2017Q3 (orig. cell 71):
the real `DATE` lives in the `SORT SEQUENCE` column, the real `TRANSCODE` in
the `DATE` column, the real `RECORDID` in the `TRANSCODE` column. The genuine
`SORT SEQUENCE` marker is destroyed for 2017Q2, so that quarter relies on the
`PURPOSE`-`TOTALS` filter only. Both the shift signature and the post-shift
`TRANSCODE` domain are asserted.

In [7]:
def adapt_B(raw: pd.DataFrame, quarter: str) -> pd.DataFrame:
    y, q = int(quarter[:4]), int(quarter[5])
    d = raw.rename(columns={"RECIP (orig.)": "VOUCHER_ID"})
    if "TRANSCODELONG" in d.columns:
        d = d.drop(columns="TRANSCODELONG")

    if quarter == "2017Q2":
        assert d["SORT SEQUENCE"].fillna("").str.match(
            r"\s*(\d\d/\d\d/\d\d)?\s*$").mean() > 0.95, \
            "2017Q2 SORT SEQUENCE no longer looks like shifted dates"
        d = d.copy()
        d["DATE"] = d["SORT SEQUENCE"]
        d["TRANSCODE"] = raw["DATE"]
        d["RECORDID"] = raw["TRANSCODE"]
        for col in ("SORT SEQUENCE", "TOTALS"):
            if col in d.columns:
                d = d.drop(columns=col)
        assert d["TRANSCODE"].fillna("").str.strip().isin(
            ["", "AP", "GL", "AR", "PY", "JV", "RC", "TR"]).mean() > 0.95, \
            "2017Q2 post-shift TRANSCODE not in valid set"

    d = _drop_total_rows(d)
    if "SORT SEQUENCE" in d.columns:
        d = d.loc[d["SORT SEQUENCE"].fillna("DETAIL") == "DETAIL"]
        d = d.drop(columns="SORT SEQUENCE")

    out = pd.DataFrame(index=d.index)
    out["YEAR"] = y
    out["QUARTER"] = f"Q{q}"
    out["BIOGUIDE_ID"] = d.get("BIOGUIDE_ID")
    out["OFFICE"] = d.get("OFFICE")
    out["PROGRAM"] = d.get("PROGRAM")
    out["CATEGORY"] = d.get("CATEGORY")
    out["PAYEE"] = d.get("PAYEE")
    out["PURPOSE"] = d.get("PURPOSE")
    out["AMOUNT"] = _to_amount(d["AMOUNT"])
    out["DATE"] = _to_txn_date(d.get("DATE"), y)
    out["START DATE"] = _to_dt(d.get("START DATE"))
    out["END DATE"] = _to_dt(d.get("END DATE"))
    out["TRANSCODE"] = d.get("TRANSCODE")
    out["RECORDID"] = d.get("RECORDID")
    out["VOUCHER_ID"] = d.get("VOUCHER_ID")
    return validate_schema(out, quarter)

### Adapter C — Schema 2B-pre (2018Q2 only)

A one-quarter mini-schema (orig. cells 73–75): `SORT SUBTOTAL DESCRIPTION` →
`CATEGORY`, `TRANSACTION DATE` → `DATE`, `DATA SOURCE` → `TRANSCODE`,
`DOCUMENT` → `RECORDID`. `YEAR`/`QUARTER` are taken from the fixed quarter,
not a column. The adapter operates only on this quarter's frame, so the
original notebook's cross-frame `.loc` index-alignment bug cannot recur.

In [8]:
def adapt_C(raw: pd.DataFrame, quarter: str) -> pd.DataFrame:
    d = raw.rename(columns={
        "SORT SUBTOTAL DESCRIPTION": "CATEGORY",
        "TRANSACTION DATE": "DATE",
        "DATA SOURCE": "TRANSCODE",
        "DOCUMENT": "RECORDID",
    })
    d = _drop_total_rows(d)
    if "SORT SEQUENCE" in d.columns:
        d = d.loc[d["SORT SEQUENCE"].fillna("DETAIL") == "DETAIL"]
        d = d.drop(columns="SORT SEQUENCE")
    out = pd.DataFrame(index=d.index)
    out["YEAR"] = 2018
    out["QUARTER"] = "Q2"
    out["BIOGUIDE_ID"] = d.get("BIOGUIDE_ID")
    out["OFFICE"] = d.get("OFFICE")
    out["PROGRAM"] = d.get("PROGRAM")
    out["CATEGORY"] = d.get("CATEGORY")
    out["PAYEE"] = d.get("PAYEE")
    out["PURPOSE"] = d.get("PURPOSE")
    out["AMOUNT"] = _to_amount(d["AMOUNT"])
    out["DATE"] = _to_txn_date(d.get("DATE"), 2018)
    out["START DATE"] = _to_dt(d.get("START DATE"))
    out["END DATE"] = _to_dt(d.get("END DATE"))
    out["TRANSCODE"] = d.get("TRANSCODE")
    out["RECORDID"] = d.get("RECORDID")
    out["VOUCHER_ID"] = d.get("VENDOR ID") if "VENDOR ID" in d.columns else pd.NA
    return validate_schema(out, quarter)

### Adapter D — Schema 2B (2022Q4 only)

The major overhaul quarter (orig. cells 81–82). `PERFORM START/END DT` →
`START/END DATE`; `DATA SOURCE` → `TRANSCODE`; `DOCUMENT` → `RECORDID`;
`VENDOR ID` → `VOUCHER_ID`. `DATE` is genuinely ISO-populated here. The
Schema-2B-only columns (`PROGRAM CODE`, `BUDGET OBJECT CLASS`, etc.) are not
in the contract and are dropped by the projection.

In [9]:
def adapt_D(raw: pd.DataFrame, quarter: str) -> pd.DataFrame:
    d = raw.rename(columns={
        "PERFORM START DT": "START DATE",
        "PERFORM END DT": "END DATE",
        "DATA SOURCE": "TRANSCODE",
        "DOCUMENT": "RECORDID",
        "VENDOR ID": "VOUCHER_ID",
    })
    d = _drop_total_rows(d)
    if "SORT SEQUENCE" in d.columns:
        d = d.loc[d["SORT SEQUENCE"].fillna("DETAIL") == "DETAIL"]
        d = d.drop(columns="SORT SEQUENCE")
    out = pd.DataFrame(index=d.index)
    out["YEAR"] = 2022
    out["QUARTER"] = "Q4"
    out["BIOGUIDE_ID"] = d.get("BIOGUIDE_ID")
    out["OFFICE"] = d.get("OFFICE")
    out["PROGRAM"] = d.get("PROGRAM")
    out["CATEGORY"] = d.get("CATEGORY")
    out["PAYEE"] = d.get("PAYEE")
    out["PURPOSE"] = d.get("PURPOSE")
    out["AMOUNT"] = _to_amount(d["AMOUNT"])
    out["DATE"] = _to_txn_date(d.get("DATE"), 2022)
    out["START DATE"] = _to_dt(d.get("START DATE"))
    out["END DATE"] = _to_dt(d.get("END DATE"))
    out["TRANSCODE"] = d.get("TRANSCODE")
    out["RECORDID"] = d.get("RECORDID")
    out["VOUCHER_ID"] = d.get("VOUCHER_ID")
    return validate_schema(out, quarter)

### Adapter E — Schema 3 (2023Q1–2024Q4 CSV + 2025Q1–Q2 XLSX)

Implements what the original notebook left unimplemented (its empty
cells 84–85). **Decision 3 (updated):** the source dropped `BIOGUIDE_ID`. The adapter emits null here; it
is populated downstream by name-matching the `ORGANIZATION` field against
`propublica_members.csv` (see the *BIOGUIDE_ID reconstruction* section below). `ORGANIZATION` → `OFFICE`, `VENDOR NAME` → `PAYEE`,
`DESCRIPTION` → `PURPOSE`, `SORT SUBTOTAL DESCRIPTION` → `CATEGORY`. Dates are
`D-Mon-YY` in the CSVs and Excel datetimes in the XLSX; both parse via the
shared helpers. The 2025Q1 file's extra `ORGANIZATION_*` columns are dropped
by the projection. Asserts `BIOGUIDE_ID` is wholly null at the adapter stage (filled later).

In [10]:
def adapt_E(raw: pd.DataFrame, quarter: str) -> pd.DataFrame:
    y, q = int(quarter[:4]), int(quarter[5])
    d = raw.rename(columns={
        "SORT SUBTOTAL DESCRIPTION": "CATEGORY",
        "TRANSACTION DATE": "DATE",
        "PERFORM START DT": "START DATE",
        "PERFORM END DT": "END DATE",
        "DESCRIPTION": "PURPOSE",
        "ORGANIZATION": "OFFICE",
        "VENDOR NAME": "PAYEE",
        "DATA SOURCE": "TRANSCODE",
        "DOCUMENT": "RECORDID",
        "VENDOR ID": "VOUCHER_ID",
    })
    d = _drop_total_rows(d)
    if "SORT SEQUENCE" in d.columns:
        d = d.loc[d["SORT SEQUENCE"].fillna("DETAIL") == "DETAIL"]
        d = d.drop(columns="SORT SEQUENCE")
    out = pd.DataFrame(index=d.index)
    out["YEAR"] = y
    out["QUARTER"] = f"Q{q}"
    out["BIOGUIDE_ID"] = pd.NA                 # DECISION 3: office-level only
    out["OFFICE"] = d.get("OFFICE")
    out["PROGRAM"] = d.get("PROGRAM")
    out["CATEGORY"] = d.get("CATEGORY")
    out["PAYEE"] = d.get("PAYEE")
    out["PURPOSE"] = d.get("PURPOSE")
    out["AMOUNT"] = _to_amount(d["AMOUNT"])
    out["DATE"] = _to_txn_date(d.get("DATE"), y)
    out["START DATE"] = _to_dt(d.get("START DATE"))
    out["END DATE"] = _to_dt(d.get("END DATE"))
    out["TRANSCODE"] = d.get("TRANSCODE")
    out["RECORDID"] = d.get("RECORDID")
    out["VOUCHER_ID"] = d.get("VOUCHER_ID")
    assert out["BIOGUIDE_ID"].isna().all(), f"{quarter}: BIOGUIDE_ID not null"
    return validate_schema(out, quarter)

## Adapter dispatch and pipeline run

`clean_quarter` reads one file and routes it to its era's adapter. The loop
runs all 62 quarters in order, records the raw row count per quarter (for the
truncation reconciliation later), and concatenates into `cleaned_df` with a
fresh `RangeIndex`.

In [11]:
ADAPTERS = {"A": adapt_A, "B": adapt_B, "C": adapt_C, "D": adapt_D, "E": adapt_E}


def clean_quarter(quarter: str, path: Path):
    raw = read_raw(path)
    out = ADAPTERS[schema_era(quarter)](raw, quarter)
    return len(raw), out


raw_counts = {}
frames = {}
for qkey in EXPECTED_QUARTERS:
    n_raw, out_df = clean_quarter(qkey, source_files[qkey])
    raw_counts[qkey] = n_raw
    frames[qkey] = out_df

cleaned_df = (
    pd.concat({q: frames[q] for q in EXPECTED_QUARTERS}, names=["YEAR-QUARTER"])
    .reset_index(level=0).reset_index(drop=True)
)
print("concatenated:", cleaned_df.shape)
cleaned_df.head()

concatenated: (6040756, 16)


,YEAR-QUARTER,YEAR,QUARTER,BIOGUIDE_ID,OFFICE,PROGRAM,CATEGORY,PAYEE,PURPOSE,AMOUNT,DATE,START DATE,END DATE,TRANSCODE,RECORDID,VOUCHER_ID
0,2011Q1,2011,Q1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"CASSIDY, ED",DIRECTOR OF HOUSE OPERATIONS,41066.67,NaT,2011-01-03,2011-03-31,NaN,NaN,"CASSIDY, ED"
1,2011Q1,2011,Q1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR,958.33,NaT,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
2,2011Q1,2011,Q1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR (OTHER COMPENSATION),13416.67,NaT,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
3,2011Q1,2011,Q1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"ELSHAMI,NADEEM",DEPUTY COMMUNICATIONS DIRECTOR,943.73,NaT,2011-01-01,2011-01-02,NaN,NaN,"ELSHAMI,NADEEM"
4,2011Q1,2011,Q1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"GREEN, JO-MARIE S",GEN COUNSEL & CHIEF OF LEG OPS,42044.44,NaT,2011-01-03,2011-03-31,NaN,NaN,"GREEN, JO-MARIE S"


## Derived columns

### CATEGORY normalization

Collapse the three spacing variants of *rent, communication, utilities* and any
residual multiple-space artifacts into one canonical value (orig. cell 15).

In [12]:
_cat_map = {
    "RENT, COMMUNICATION, UTILITIES": "RENT COMMUNICATION UTILITIES",
    "RENT  COMMUNICATION  UTILITIES": "RENT COMMUNICATION UTILITIES",
}
cleaned_df["CATEGORY"] = (
    cleaned_df["CATEGORY"].replace(_cat_map)
    .str.replace(r"\s+", " ", regex=True).str.strip()
)
sorted(cleaned_df["CATEGORY"].dropna().unique())

['BENEFITS TO FORMER PERSONNEL',
 'EQUIPMENT',
 'FRANKED MAIL',
 'INSURANCE CLAIMS & INDEMNITIES',
 'OTHER SERVICES',
 'PERSONNEL BENEFITS',
 'PERSONNEL COMPENSATION',
 'PRINTING AND REPRODUCTION',
 'RENT COMMUNICATION UTILITIES',
 'SUPPLIES AND MATERIALS',
 'TRANSPORTATION OF THINGS',
 'TRAVEL']

### TERM_QUARTER

A U.S. House Congress spans two calendar years (eight quarters), beginning in
an odd year. `TERM_QUARTER` is the 1–8 position within the Congress, derived
from the calendar year/quarter — replacing the original notebook's fragile
positional `i % 8` map (which was keyed to whatever quarter happened to be
first in the data).

In [13]:
def _term_quarter(row) -> int:
    y, q = int(row["YEAR"]), int(row["QUARTER"][1])
    congress_start = y if y % 2 == 1 else y - 1
    return (y - congress_start) * 4 + q          # 1..8


cleaned_df["TERM_QUARTER"] = (
    cleaned_df.apply(_term_quarter, axis=1).astype("int8")
)
cleaned_df["TERM_QUARTER"].value_counts().sort_index()

TERM_QUARTER
1    890025
2    855951
3    726508
4    716328
5    779193
6    707767
7    691461
8    673523
Name: count, dtype: int64

### PERIOD_DATE and DATE_IS_RECONSTRUCTED (Decision 5)

`DATE` is the true transaction date, null where the source has none (notably
personnel-compensation rows). `PERIOD_DATE` falls back to `START DATE` so
time-series analyses always have a date; `DATE_IS_RECONSTRUCTED` flags those
rows. The `DATE − START DATE` pre-billing audit in `analysis.ipynb` keeps using
the true (possibly null) `DATE`, so that signal is preserved.

In [14]:
cleaned_df["PERIOD_DATE"] = cleaned_df["DATE"].fillna(cleaned_df["START DATE"])
cleaned_df["DATE_IS_RECONSTRUCTED"] = (
    cleaned_df["DATE"].isna() & cleaned_df["PERIOD_DATE"].notna()
)
cleaned_df["YEAR"] = cleaned_df["YEAR"].astype("int16")
cleaned_df = validate_schema(cleaned_df, "ALL", COMMON_COLUMNS)
print("DATE reconstructed fraction:",
      round(cleaned_df["DATE_IS_RECONSTRUCTED"].mean(), 3))
cleaned_df.head()

DATE reconstructed fraction: 0.151


,YEAR-QUARTER,YEAR,QUARTER,TERM_QUARTER,BIOGUIDE_ID,OFFICE,PROGRAM,CATEGORY,PAYEE,PURPOSE,AMOUNT,DATE,PERIOD_DATE,DATE_IS_RECONSTRUCTED,START DATE,END DATE,TRANSCODE,RECORDID,VOUCHER_ID
0,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"CASSIDY, ED",DIRECTOR OF HOUSE OPERATIONS,41066.67,NaT,2011-01-03,True,2011-01-03,2011-03-31,NaN,NaN,"CASSIDY, ED"
1,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR,958.33,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
2,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR (OTHER COMPENSATION),13416.67,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
3,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"ELSHAMI,NADEEM",DEPUTY COMMUNICATIONS DIRECTOR,943.73,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"ELSHAMI,NADEEM"
4,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,<NA>,PERSONNEL COMPENSATION,"GREEN, JO-MARIE S",GEN COUNSEL & CHIEF OF LEG OPS,42044.44,NaT,2011-01-03,True,2011-01-03,2011-03-31,NaN,NaN,"GREEN, JO-MARIE S"


### BIOGUIDE_ID reconstruction (Decision 3, updated)

Schema 3 (2023Q1–2025Q2) dropped `BIOGUIDE_ID` from the source. Recover it by parsing the `OFFICE` text (Schema 3's `ORGANIZATION`, e.g.
`2023 HON. KRISTEN MCDONALD RIVET`) into `(first, last)` and looking it up against `propublica_members.csv`. Propublica is deduplicated by `bioguide_id` first (the file lists one row per Congress-of-service); names that collapse to more than one bioguide after dedupe are left null and reported, per Decision 3.

**Normalization.** Accents folded (NFKD), suffixes (`JR.`, `SR.`, `II`–`V`) stripped, quoted nicknames (`"BUDDY"`) removed, single-letter middle initials with periods (`B.`, `G.K.`) dropped. Office strings that don't fit `YYYY HON. <NAME>` (`OFFICE OF THE SPEAKER`, `CHIEF ADMIN OFCR`, `REPUBLICAN CONFERENCE`, …) and district placeholders (`2023 HON. 4TH DISTRICT OF VIRGINIA`) intentionally stay null — they have no member identity to resolve.

**Source.** [ProPublica Represent](https://projects.propublica.org/represent/) member roster, mirrored locally as `propublica_members.csv`.

In [ ]:
import unicodedata

PROPUBLICA_PATH = Path("propublica_members.csv")

_HON_RE         = re.compile(r"^\s*\d{4}\s+HON\.\s+(.+?)\s*$")
_DISTRICT_RE    = re.compile(r"^\d+\s*(ST|ND|RD|TH)?\s+DISTRICT\b", re.I)
_SUFFIX_RE      = re.compile(r"\s+(JR|SR|II|III|IV|V)\.?\s*$", re.I)
_QUOTED_RE      = re.compile(r'"[^"]*"')
_INITIAL_TOK_RE = re.compile(r"^(?:[A-Z]\.)+$")


def _fold(s):
    """Uppercase, strip accents, collapse whitespace. None/NaN-safe."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).upper().strip()


def _parse_member_name(office):
    """Parse OFFICE → (FIRST, LAST_FULL, LAST_TOKEN) or None.

    LAST_FULL is the joined remainder ("RAY CISNEROS"); LAST_TOKEN is the
    final word ("CISNEROS"). Returning both lets the matcher try the strict
    form first and fall back when propublica drops middle names.
    """
    if not isinstance(office, str):
        return None
    m = _HON_RE.match(office)
    if not m:
        return None
    inner = m.group(1)
    if _DISTRICT_RE.match(inner):
        return None
    inner = _QUOTED_RE.sub(" ", inner)               # drop "BUDDY"
    inner = _SUFFIX_RE.sub("", inner)                # drop trailing JR/III
    inner = _fold(inner)
    tokens = [t for t in inner.split() if not _INITIAL_TOK_RE.match(t)]
    if len(tokens) < 2:
        return None
    return tokens[0], " ".join(tokens[1:]), tokens[-1]


_pp = (
    pd.read_csv(PROPUBLICA_PATH)
      .drop_duplicates(subset=["bioguide_id"])
)
_pp["_first"] = _pp["first_name"].map(_fold)
_pp["_last"]  = _pp["last_name"].map(_fold)
_pp["_last_token"] = _pp["_last"].str.split().str[-1]

# Primary: exact (first, full-last). Secondary: (first, last-token) only when unique.
_g_full = _pp.groupby(["_first", "_last"])["bioguide_id"].apply(list)
_LOOKUP_FULL = {k: v[0] for k, v in _g_full.items() if len(v) == 1}
_AMBIG_FULL  = {k for k, v in _g_full.items() if len(v) > 1}

_g_tok = _pp.groupby(["_first", "_last_token"])["bioguide_id"].apply(list)
_LOOKUP_TOK  = {k: v[0] for k, v in _g_tok.items() if len(v) == 1}


def _resolve(parsed):
    if parsed is None:
        return None
    first, last_full, last_tok = parsed
    bid = _LOOKUP_FULL.get((first, last_full))
    if bid:
        return bid
    return _LOOKUP_TOK.get((first, last_tok))


# Apply only to Schema 3 rows (2023+); adapter emitted null there.
schema3_mask = cleaned_df["YEAR"] >= 2023
parsed  = cleaned_df.loc[schema3_mask, "OFFICE"].map(_parse_member_name)
matched = parsed.map(_resolve)
cleaned_df.loc[schema3_mask, "BIOGUIDE_ID"] = matched.values

# Office-level diagnostics (one row per unique OFFICE string)
offices = cleaned_df.loc[schema3_mask, "OFFICE"].dropna().unique()
parsed_offices = {o: _parse_member_name(o) for o in offices}
member     = [o for o, p in parsed_offices.items() if p is not None]
non_member = [o for o, p in parsed_offices.items() if p is None]
resolved   = [o for o, p in parsed_offices.items() if _resolve(p) is not None]
unmatched  = [o for o in member if _resolve(parsed_offices[o]) is None]

print(f"Schema-3 unique OFFICEs: {len(offices)}")
print(f"  member-style (YYYY HON. ...): {len(member)}")
print(f"    resolved to bioguide      : {len(resolved)}")
print(f"    unresolved                : {len(unmatched)}")
print(f"  non-member / vacancy        : {len(non_member)}")
print(f"Row-level BIOGUIDE_ID present 2023+: "
      f"{cleaned_df.loc[schema3_mask, 'BIOGUIDE_ID'].notna().mean():.1%}")


## Integrity checks

Each cell is an asserted invariant; on `Restart & Run All` any regression
fails loudly here rather than silently corrupting a downstream finding.

**1. Quarter coverage & power-of-two truncation watch.** Every cleaned quarter is non-empty, and `raw − dropped == clean` per quarter (catches silent read truncation). No suspicious power-of-two row counts.

In [15]:
per_q = cleaned_df.groupby("YEAR-QUARTER").size()
assert set(per_q.index) == set(EXPECTED_QUARTERS)
assert (per_q > 0).all(), "an empty quarter slipped through"
assert cleaned_df["YEAR"].min() == 2011, "2010 leaked into the cleaned universe"
assert not set(cleaned_df["YEAR-QUARTER"]) & set(EXCLUDED_QUARTERS)
SUSPECT = {65536, 1048576, 16384, 262144, 524288}
assert not (set(raw_counts.values()) & SUSPECT), "power-of-two raw count"
assert not (set(per_q.values) & SUSPECT), "power-of-two clean count"
dropped_total = sum(raw_counts.values()) - len(cleaned_df)
print(f"raw={sum(raw_counts.values()):,}  clean={len(cleaned_df):,}  "
      f"removed(subtotals/totals)={dropped_total:,}")

raw=6,233,106  clean=6,040,756  removed(subtotals/totals)=192,350


**2. AMOUNT is numeric and essentially complete.**

In [16]:
assert cleaned_df["AMOUNT"].dtype == np.float64
na_frac = cleaned_df["AMOUNT"].isna().mean()
assert na_frac < 1e-4, f"too many non-numeric AMOUNT ({na_frac:.6%})"
assert np.isfinite(cleaned_df["AMOUNT"].abs().max())
print(f"AMOUNT null fraction: {na_frac:.8%};  "
      f"grand total: ${cleaned_df['AMOUNT'].sum():,.2f}")

AMOUNT null fraction: 0.00001655%;  grand total: $20,174,379,954.18


**3. Dates are typed and within a plausible window.** `START DATE` ranges back to the late 1990s (long-running obligations are real); `DATE` is the posting date.

In [17]:
for c in ("DATE", "PERIOD_DATE", "START DATE", "END DATE"):
    assert pd.api.types.is_datetime64_any_dtype(cleaned_df[c]), c
assert cleaned_df["START DATE"].dropna().dt.year.between(1995, 2035).all()
assert cleaned_df["DATE"].dropna().dt.year.between(2009, 2026).all()
print("DATE range       :", cleaned_df["DATE"].min(), "->",
      cleaned_df["DATE"].max())
print("PERIOD_DATE range:", cleaned_df["PERIOD_DATE"].min(), "->",
      cleaned_df["PERIOD_DATE"].max())
print("DATE reconstructed (fallback to START DATE):",
      f"{cleaned_df['DATE_IS_RECONSTRUCTED'].mean():.1%}")

DATE range       : 2011-01-01 00:00:00 -> 2025-06-30 00:00:00
PERIOD_DATE range: 2005-12-01 00:00:00 -> 2025-06-30 00:00:00
DATE reconstructed (fallback to START DATE): 15.1%


**4. BIOGUIDE_ID coverage.** Substantially present for 2011–2022 (member rows from the source); for 2023Q1–2025Q2 the column is reconstructed from `ORGANIZATION` via propublica name-match, so a non-trivial share of rows now carries a bioguide too.

In [18]:
post = cleaned_df["YEAR"] >= 2023
mid = cleaned_df["YEAR"].between(2011, 2022)
mid_rate  = cleaned_df.loc[mid,  "BIOGUIDE_ID"].notna().mean()
post_rate = cleaned_df.loc[post, "BIOGUIDE_ID"].notna().mean()
assert mid_rate  > 0.5, f"member rows missing 2011-2022: {mid_rate:.1%}"
assert post_rate > 0.5, \
    f"propublica match recovered too few 2023+ bioguides: {post_rate:.1%}"
print(f"BIOGUIDE_ID present 2011-2022 (source) : {mid_rate:.1%}")
print(f"BIOGUIDE_ID present 2023+    (matched): {post_rate:.1%}")


BIOGUIDE_ID present 2011-2022: 80.2%
BIOGUIDE_ID present 2023+   : 0.0%


**5. CATEGORY consistency & totals removal.** The category set is small and stable; the three rent variants collapsed to one; no subtotal/total rows survived.

In [19]:
cats = set(cleaned_df["CATEGORY"].dropna().unique())
assert "RENT COMMUNICATION UTILITIES" in cats
assert not {"RENT, COMMUNICATION, UTILITIES",
            "RENT  COMMUNICATION  UTILITIES"} & cats
assert len(cats) <= 20, f"unexpected category sprawl: {sorted(cats)}"
assert not cleaned_df["PURPOSE"].fillna("").str.contains("TOTALS").any()
assert "SORT SEQUENCE" not in cleaned_df.columns
print(f"{len(cats)} categories:", sorted(cats))

12 categories: ['BENEFITS TO FORMER PERSONNEL', 'EQUIPMENT', 'FRANKED MAIL', 'INSURANCE CLAIMS & INDEMNITIES', 'OTHER SERVICES', 'PERSONNEL BENEFITS', 'PERSONNEL COMPENSATION', 'PRINTING AND REPRODUCTION', 'RENT COMMUNICATION UTILITIES', 'SUPPLIES AND MATERIALS', 'TRANSPORTATION OF THINGS', 'TRAVEL']


**6. Schema-shift correctness (2017Q2).** After the positional fix, `TRANSCODE` is in the valid posting-code domain and `DATE` parses.

In [20]:
q2 = cleaned_df[cleaned_df["YEAR-QUARTER"] == "2017Q2"]
assert len(q2) > 50_000, f"2017Q2 lost its rows: {len(q2)}"
assert q2["TRANSCODE"].fillna("").str.strip().isin(
    ["", "AP", "GL", "AR", "PY", "JV", "RC", "TR"]).mean() > 0.95
assert q2["DATE"].notna().mean() > 0.5
print("2017Q2 rows:", len(q2),
      "| TRANSCODE:", q2["TRANSCODE"].value_counts().head(3).to_dict())

2017Q2 rows: 98669 | TRANSCODE: {'AP': 73377, 'GL': 12995, '   ': 11956}


## Pinned headline numbers

These exact figures are the reproducibility contract. If a re-download of the
source changes them, this cell fails on purpose — update the pins here with a
documented reason rather than letting a silent change propagate downstream.

In [21]:
EXPECTED = {
    "n_quarters": 58,
    "raw_rows": 6_233_106,
    "clean_rows": 6_040_756,
    "removed_rows": 192_350,
    "amount_total": 20_174_379_954.18,
}
actual = {
    "n_quarters": cleaned_df["YEAR-QUARTER"].nunique(),
    "raw_rows": sum(raw_counts.values()),
    "clean_rows": len(cleaned_df),
    "removed_rows": sum(raw_counts.values()) - len(cleaned_df),
    "amount_total": round(float(cleaned_df["AMOUNT"].sum()), 2),
}
for k in EXPECTED:
    assert actual[k] == EXPECTED[k], f"{k}: {actual[k]} != {EXPECTED[k]}"
print("pinned headline numbers OK:", actual)

pinned headline numbers OK: {'n_quarters': 58, 'raw_rows': 6233106, 'clean_rows': 6040756, 'removed_rows': 192350, 'amount_total': 20174379954.18}


## Write cleaned output

Write the single artifact `analysis.ipynb` consumes (Git-LFS-tracked), then
re-read it and assert the round trip preserves shape, columns, and the date
dtypes — proving the file the trust-contract link lands on is the file this
notebook produced.

In [22]:
cleaned_df.to_csv(OUTPUT_PATH, index=False, encoding="utf8")

_check = pd.read_csv(
    OUTPUT_PATH, low_memory=False,
    parse_dates=["DATE", "PERIOD_DATE", "START DATE", "END DATE"],
)
assert list(_check.columns) == COMMON_COLUMNS
assert len(_check) == len(cleaned_df)
for c in ("DATE", "PERIOD_DATE", "START DATE", "END DATE"):
    assert pd.api.types.is_datetime64_any_dtype(_check[c]), c
print(f"wrote {OUTPUT_PATH} — {len(_check):,} rows x {len(_check.columns)} cols")

wrote all_disbursements.csv — 6,040,756 rows x 19 cols
